In [ ]:
import sys,json
from pathlib import Path
cwd=Path().resolve(); repo_root=cwd.parent if cwd.name=='notebooks' else cwd
sys.path.insert(0,str(repo_root/'src'))
from dotenv import load_dotenv; load_dotenv(repo_root/'.env',override=True)

In [ ]:
from rag.retrieval.vector_retriever import VectorRetriever
ret = VectorRetriever()
queries = [
    'maternal mortality Nigeria',              # semantic → vector wins
    'DHS 2022 Kenya FR380',                    # exact terms → BM25 wins
    'PR157 contraceptive prevalence',          # filename → BM25 wins
    'fertility rate sub-Saharan Africa 2021',  # mixed → hybrid wins
]
for q in queries:
    docs = ret.retrieve(q, top_k=3)
    print(f'Query: {q}')
    for d in docs:
        print(f'  [{d.metadata["similarity_score"]:.3f}] {d.metadata.get("country","")} {d.metadata.get("year","")} {d.page_content[:80]}...')
    print()

In [ ]:
from rag.ingestion.loader import load_directory
from rag.ingestion.cleaner import clean_pages
from rag.ingestion.chunker import ChunkStrategy, chunk_pages
from rag.retrieval.bm25_retriever import BM25Retriever
meta_map = json.loads((repo_root/'data'/'metadata.json').read_text())
pages   = load_directory(repo_root/'data'/'raw', metadata_map=meta_map)
cleaned = clean_pages(pages)
chunks  = chunk_pages(cleaned, ChunkStrategy.RECURSIVE)
bm25    = BM25Retriever(chunks)
print(f'BM25 index built over {len(chunks):,} documents')

In [ ]:
query = 'DHS 2022 Kenya FR380'
print(f'BM25 results for: {query}')
for d in bm25.retrieve(query, top_k=5):
    print(f'  score={d.metadata["bm25_score"]:.3f} | {d.metadata.get("country","")} {d.metadata.get("file_name","")}')
    print(f'    {d.page_content[:120]}...')
    print()

In [ ]:
from rag.retrieval.hybrid_retriever import HybridRetriever, reciprocal_rank_fusion
ret    = VectorRetriever()
hybrid = HybridRetriever(ret, bm25)
query  = 'DHS 2022 Kenya FR380 contraceptive prevalence'
results = hybrid.retrieve(query)
print(f'Hybrid results: {len(results)}')
for d in results[:5]:
    print(f'  rrf={d.metadata["rrf_score"]:.6f} | {d.metadata.get("country","")} {d.metadata.get("year","")}')
    print(f'    {d.page_content[:120]}...')
    print()

In [ ]:
from rag.retrieval.hybrid_retriever import reciprocal_rank_fusion
vec_docs = ret.retrieve('maternal mortality Kenya', top_k=10)
bm25_docs = bm25.retrieve('maternal mortality Kenya', top_k=10)
fused = reciprocal_rank_fusion([vec_docs, bm25_docs], k=60, top_n=5)
print('RRF formula: score = Σ 1/(k + rank_i)  where k=60')
print()
print('Vector top-3:')
for d in vec_docs[:3]: print(f'  {d.metadata.get("country","")} | {d.page_content[:80]}...')
print()
print('BM25 top-3:')
for d in bm25_docs[:3]: print(f'  {d.metadata.get("country","")} | {d.page_content[:80]}...')
print()
print('RRF fused top-3:')
for d in fused[:3]: print(f'  rrf={d.metadata["rrf_score"]:.6f} | {d.metadata.get("country","")} | {d.page_content[:80]}...')

## ✅ Episode 6 complete

**Episode 7:** Metadata filtering — let the LLM extract country/year from the query automatically.